In [1]:
# Google Colab setup: fetch repository and set working directory
from pathlib import Path
import os

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists():
    !git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming

NOTEBOOK_DIR = REPO_ROOT / 'module_2/week_6/use_case_4'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')

Cloning into '/content/BITS_programming'...
remote: Enumerating objects: 2101, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 2101 (delta 11), reused 15 (delta 3), pack-reused 2066 (from 1)
Receiving objects: 100% (2101/2101), 263.06 MiB | 16.97 MiB/s, done.
Resolving deltas: 100% (400/400), done.
Updating files: 100% (1348/1348), done.
Working directory: /content/BITS_programming/module_2/week_6/use_case_4


In [2]:
# AWS credentials setup via Google Colab Secrets
import os

def get_colab_secret(name, required=True):
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception as exc:
        if required:
            raise RuntimeError(f"Unable to read Colab Secret: {name}") from exc
        return None
    if required and (value is None or value == ""):
        raise RuntimeError(f"Add the Colab Secret {name} and grant this notebook access.")
    return value

AWS_ACCESS_KEY_ID = get_colab_secret("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = get_colab_secret("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = get_colab_secret("AWS_SESSION_TOKEN", required=False)

os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
if AWS_SESSION_TOKEN:
    os.environ["AWS_SESSION_TOKEN"] = AWS_SESSION_TOKEN

_aws_region = os.getenv("AWS_REGION") or os.getenv("AWS_DEFAULT_REGION") or "ap-south-1"
os.environ["AWS_REGION"] = _aws_region
os.environ["AWS_DEFAULT_REGION"] = _aws_region

print(f"AWS credentials loaded. Region: {_aws_region}")

AWS credentials loaded. Region: ap-south-1


In [3]:
# Install boto3 and provision dynamic S3 buckets using AWS Account ID
!pip install boto3 pandas numpy -q
import os
import boto3
from pathlib import Path
from botocore.exceptions import ClientError

region = os.environ.get("AWS_REGION", "ap-south-1")
sts_client = boto3.client("sts", region_name=region)
account_id = sts_client.get_caller_identity()["Account"]

_s3 = boto3.client("s3", region_name=region)

BASE_BUCKET_NAMES = ['usecase-etl-1', 'usecase-etl-2']
REQUIRED_BUCKETS = [f"{b}-{account_id}" for b in BASE_BUCKET_NAMES]

for _b in REQUIRED_BUCKETS:
    try:
        if region == "us-east-1":
            _s3.create_bucket(Bucket=_b)
        else:
            _s3.create_bucket(
                Bucket=_b,
                CreateBucketConfiguration={"LocationConstraint": region}
            )
        print(f"Created bucket: {_b}")
    except ClientError as _e:
        _code = _e.response.get("Error", {}).get("Code", "")
        if _code in ("BucketAlreadyExists", "BucketAlreadyOwnedByYou", "Conflict"):
            print(f"Bucket already exists (reusing existing): {_b}")
        else:
            print(f"Could not create bucket {_b} ({_code}): {_e}")

# Auto-seed datasets to S3 if missing
BUCKET_1 = REQUIRED_BUCKETS[0]
REPO_ROOT = Path('/content/BITS_programming')

DATASET_MAP = {
    "churn/ml_ready/train.csv": [Path("../04_Datasets/baseline/train.csv"), REPO_ROOT / "04_Datasets/baseline/train.csv"],
    "churn/monitoring/incoming/future_scoring_sample.csv": [Path("../04_Datasets/monitoring/future_scoring_sample.csv"), REPO_ROOT / "04_Datasets/monitoring/future_scoring_sample.csv"]
}

for key, paths in DATASET_MAP.items():
    local_file = next((p for p in paths if p.exists()), None)
    try:
        _s3.head_object(Bucket=BUCKET_1, Key=key)
        print(f"ℹ️ S3 object already present: s3://{BUCKET_1}/{key}")
    except Exception:
        if local_file:
            print(f"📦 Uploading local file ({local_file}) to s3://{BUCKET_1}/{key}...")
            _s3.upload_file(str(local_file), BUCKET_1, key)
            print(f"✅ Uploaded: s3://{BUCKET_1}/{key}")
        else:
            print(f"⚠️ Local file missing for key: {key}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 6.8 MB/s eta 0:00:00
Bucket already exists (reusing existing): usecase-etl-1-455865672536
Bucket already exists (reusing existing): usecase-etl-2-455865672536
⚠️ Local file missing for key: churn/ml_ready/train.csv
⚠️ Local file missing for key: churn/monitoring/incoming/future_scoring_sample.csv


# Use Case 4 - ETL for Monitoring and Model Review on AWS

This lab treats monitoring as another **ETL-style batch workflow**.

## What this lab covers
- Amazon S3 monitoring input zone
- Baseline profiling and incoming batch comparison
- Bias or explainability discussion in SageMaker
- Decision ladder: retain, review, retrain, rollback

## ETL flow in one line
`Extract baseline and incoming data -> Transform into comparable profiles -> Load findings and review artifacts -> Decide lifecycle action`

In [4]:
import pandas as pd, numpy as np, json, boto3, io
from pathlib import Path

REPO_ROOT = Path('/content/BITS_programming')
region = os.environ.get("AWS_REGION", "ap-south-1")
sts_client = boto3.client("sts", region_name=region)
account_id = sts_client.get_caller_identity()["Account"]
BUCKET_1 = f"usecase-etl-1-{account_id}"

# Resolve local dataset paths or read directly from S3
train_path = next((p for p in [
    Path("../04_Datasets/baseline/train.csv"),
    REPO_ROOT / "04_Datasets" / "baseline" / "train.csv",
] + list(REPO_ROOT.glob("**/train.csv")) if p.exists()), None)

future_path = next((p for p in [
    Path("../04_Datasets/monitoring/future_scoring_sample.csv"),
    REPO_ROOT / "04_Datasets" / "monitoring" / "future_scoring_sample.csv",
] + list(REPO_ROOT.glob("**/future_scoring_sample.csv")) if p.exists()), None)

s3_client = boto3.client("s3", region_name=region)

if train_path:
    train_df = pd.read_csv(train_path)
else:
    obj = s3_client.get_object(Bucket=BUCKET_1, Key="churn/ml_ready/train.csv")
    train_df = pd.read_csv(io.BytesIO(obj['Body'].read()))

if future_path:
    future_df = pd.read_csv(future_path)
else:
    obj = s3_client.get_object(Bucket=BUCKET_1, Key="churn/monitoring/incoming/future_scoring_sample.csv")
    future_df = pd.read_csv(io.BytesIO(obj['Body'].read()))

print(f"Baseline Train DF Shape: {train_df.shape}")
print(f"Monitoring Future DF Shape: {future_df.shape}")

Baseline Train DF Shape: (520, 25)
Monitoring Future DF Shape: (130, 24)


## 1. Extract the baseline and incoming batch
In the live AWS demo, show the baseline in `churn/ml_ready/` and the incoming batch in `churn/monitoring/incoming/`.

## 2. Transform into monitoring profiles

In [5]:
baseline_rows = []
for col in train_df.columns:
    if pd.api.types.is_numeric_dtype(train_df[col]):
        baseline_rows.append({"column_name": col, "column_type": "numeric", "mean": train_df[col].mean(), "std": train_df[col].std(), "null_rate": train_df[col].isna().mean()})
    else:
        baseline_rows.append({"column_name": col, "column_type": "categorical", "top_value": train_df[col].astype(str).fillna("MISSING").value_counts().index[0], "null_rate": train_df[col].isna().mean()})
baseline_profile = pd.DataFrame(baseline_rows)
baseline_profile.head()

,column_name,column_type,top_value,null_rate,mean,std
0,customerID,categorical,7590-0639,0.0,NaN,NaN
1,gender,categorical,Male,0.0,NaN,NaN
2,SeniorCitizen,numeric,NaN,0.0,0.190385,0.392983
3,Partner,categorical,No,0.0,NaN,NaN
4,Dependents,categorical,Yes,0.0,NaN,NaN


In [6]:
# Ensure target artifacts directory exists before saving
artifacts_dir = Path("../05_Artifacts")
if not artifacts_dir.parent.exists():
    artifacts_dir = REPO_ROOT / "module_2/week_6/05_Artifacts"
artifacts_dir.mkdir(parents=True, exist_ok=True)

baseline_profile.to_csv(artifacts_dir / "baseline_profile_generated.csv", index=False)
print(f"Saved profile artifact to {artifacts_dir / 'baseline_profile_generated.csv'}")

Saved profile artifact to ../05_Artifacts/baseline_profile_generated.csv


## 3. Compare incoming data against the baseline

In [7]:
findings = []
for col in future_df.columns:
    if col not in train_df.columns:
        continue
    if pd.api.types.is_numeric_dtype(train_df[col]) and pd.api.types.is_numeric_dtype(future_df[col]):
        b = train_df[col].mean(); i = future_df[col].mean()
        pct = ((i-b)/b) if b else np.nan
        findings.append({"column_name": col, "check_type": "mean_shift", "pct_change": pct, "flag": abs(pct) > 0.10 if pd.notna(pct) else False})
    else:
        findings.append({"column_name": col, "check_type": "top_category_shift", "pct_change": np.nan, "flag": train_df[col].astype(str).mode()[0] != future_df[col].astype(str).mode()[0]})
findings_df = pd.DataFrame(findings)
findings_df.head(20)

,column_name,check_type,pct_change,flag
0,customerID,top_category_shift,NaN,True
1,gender,top_category_shift,NaN,True
2,SeniorCitizen,mean_shift,0.010101,False
3,Partner,top_category_shift,NaN,False
4,Dependents,top_category_shift,NaN,False
5,tenure,mean_shift,0.028907,False
6,PhoneService,top_category_shift,NaN,False
7,MultipleLines,top_category_shift,NaN,False
8,InternetService,top_category_shift,NaN,False
9,OnlineSecurity,top_category_shift,NaN,False


In [8]:
flagged = findings_df[findings_df["flag"] == True]
flagged.to_csv(artifacts_dir / "flagged_drift_findings_generated.csv", index=False)
print(f"Saved drift findings to {artifacts_dir / 'flagged_drift_findings_generated.csv'}")
flagged

Saved drift findings to ../05_Artifacts/flagged_drift_findings_generated.csv


,column_name,check_type,pct_change,flag
0,customerID,top_category_shift,NaN,True
1,gender,top_category_shift,NaN,True
23,avg_monthly_spend_gap,mean_shift,1.46834,True


## 4. Bias and explainability talking point
Use SageMaker Clarify or a simple segmented review to show that customer groups should be checked, not just model accuracy.

In [9]:
if "label" in train_df.columns and "Contract" in train_df.columns:
    segment = train_df.groupby("Contract")["label"].mean().reset_index().rename(columns={"label": "avg_churn_rate"})
else:
    segment = train_df.head()
segment

,Contract,avg_churn_rate
0,Month-to-month,0.363344
1,One year,0.235849
2,Two year,0.067961


## 5. Optional AWS Glue bridge
Open `06_Assets/code/optional_glue_job_uc4.py` to show how monitoring profiles can be created with a batch ETL job.

## 6. Decision ladder
Minor findings keep the model active. Larger findings trigger review, retraining, or rollback.

In [10]:
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))

Current Time in IST: 2026-09-14 12:28:02
